# Lab | Tools prompting


## Intro

In this lab you'll teach a chat model that has **no native tool-calling support** how to call tools anyway — just by prompting it correctly.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: defining `add` and `multiply` tools, writing a prompt that describes them to the model, parsing the model's JSON tool-call output, and invoking the right tool with the right arguments.
2. **Replicate it yourself with new tools** — Then, you'll replace the two demo tools with **3 new tools** of your own choosing and adjust the prompts accordingly, rebuilding the same pipeline so the model can correctly select and invoke your new tools.

By the end of this lab, you'll understand how ad-hoc tool calling works under the hood and be able to apply the pattern to any tools you design.

<br>

## How to add ad-hoc tool calling capability to LLMs and Chat Models

Some models have been fine-tuned for tool calling and provide a dedicated API for tool calling. Generally, such models are better at tool calling than non-fine-tuned models, and are recommended for use cases that require tool calling. Please see the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide for more information.

In this guide, we'll see how to add **ad-hoc** tool calling support to a chat model. This is an alternative method to invoke tools if you're using a model that does not natively support tool calling.

<br>

We'll do this by simply writing a prompt that will get the model to invoke the appropriate tools. Here's a diagram of the logic:

<br>

![chain](https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/tool_chain.svg)

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
# Dependencies used below (already installed in this environment):
# %pip install langchain-core nbformat nbclient ipykernel

<br>

## Initial Setup



You can select any of the given models for this how-to guide. Keep in mind that most of these models already [support native tool calling](https://python.langchain.com/docs/integrations/chat), so using the prompting strategy shown here doesn't make sense for these models, and instead you should follow the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide.

To illustrate the idea, we'll use `phi3` via Ollama, which does **NOT** have native support for tool calling. If you'd like to use `Ollama` as well follow [these instructions](https://python.langchain.com/docs/integrations/chat/ollama).

In [2]:
import json
import os
from urllib.request import Request, urlopen
from langchain_core.runnables import RunnableLambda

# Use the locally installed model. Set OLLAMA_MODEL=phi3 to use the demo model.
# We send plain text and never use the native tool-calling API.
MODEL_NAME = os.environ.get("OLLAMA_MODEL", "qwen3:4b")
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")

def generate_text(prompt_value):
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt_value.to_string(),
        "stream": False,
        "think": False,
        "format": "json",
        "options": {"temperature": 0, "num_predict": 256},
    }
    request = Request(
        f"{OLLAMA_URL}/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urlopen(request, timeout=300) as response:
        return json.load(response)["response"]

model = RunnableLambda(generate_text)
print(f"Using {MODEL_NAME} via Ollama, with prompt-based tool selection.")

Using qwen3:4b via Ollama, with prompt-based tool selection.


<br>

##  How to Install and Run Ollama with the Phi-3 Model

This guide walks you through installing **Ollama** and running the **Phi-3** model on Windows, macOS, and Linux.

---

### Windows

1. **Download Ollama for Windows**  
   Go to: [https://ollama.com/download](https://ollama.com/download)  
   Download and run the installer.

2. **Verify Installation**  
   Open **Command Prompt** and type:
   ```bash
   ollama --version
   ```

3. **Run the Phi-3 Model**  
   In the same terminal:
   ```bash
   ollama run phi3
   ```

4. **If you get a CUDA error (GPU memory issue)**  
   Run Ollama in **CPU mode**:
   ```bash
   set OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

###  macOS

1. **Install via Homebrew**  
   Open the Terminal and run:
   ```bash
   brew install ollama
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

###  Linux

1. **Install Ollama**  
   Open a terminal and run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

###  Notes

- The first time you run `ollama run phi3`, it will **download the model**, so make sure you’re connected to the internet.
- Once downloaded, it works **offline**.
- Keep the terminal open and running in the background while using Ollama from your code or notebook.


<br>

## Create a tool

First, let's create an `add` and `multiply` tools. For more information on creating custom tools, please see [this guide](https://python.langchain.com/docs/how_to/custom_tools/).

In [3]:
from langchain_core.tools import tool


@tool
def multiply(x: float, y: float) -> float:
    """Multiply two numbers together."""
    return x * y


@tool
def add(x: int, y: int) -> int:
    "Add two numbers."
    return x + y


tools = [multiply, add]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

--
multiply
Multiply two numbers together.
{'x': {'title': 'X', 'type': 'number'}, 'y': {'title': 'Y', 'type': 'number'}}
--
add
Add two numbers.
{'x': {'title': 'X', 'type': 'integer'}, 'y': {'title': 'Y', 'type': 'integer'}}


In [4]:
multiply.invoke({"x": 4, "y": 5})

20.0

<br>

## Creating our prompt

We'll want to write a prompt that specifies the tools the model has access to, the arguments to those tools, and the desired output format of the model. In this case we'll instruct it to output a JSON blob of the form `{"name": "...", "arguments": {...}}`.

In [5]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)

multiply(x: float, y: float) -> float - Multiply two numbers together.
add(x: int, y: int) -> int - Add two numbers.


In [6]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools. 
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use. 
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding 
to the argument names and the values corresponding to the requested values.
"""

prompt = ChatPromptTemplate.from_messages(
    [ ("system", "{tool_instructions}"), ("user", "{input}")]
).partial(tool_instructions=system_prompt)

In [7]:
chain = prompt | model
message = chain.invoke({"input": "what's 3 plus 1132"})

# Let's take a look at the output from the model
# if the model is an LLM (not a chat model), the output will be a string.
if isinstance(message, str):
    print(message)
else:  # Otherwise it's a chat model
    print(message.content)

{
  "name": "add",
  "arguments": {
    "x": 3,
    "y": 1132
  }
}


<br>

## Adding an output parser

We'll use the `JsonOutputParser` for parsing our models output to JSON.

In [8]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt | model | JsonOutputParser()
chain.invoke({"input": "what's thirteen times 4"})

{'name': 'multiply', 'arguments': {'x': 13, 'y': 4}}

<br>

Amazing! 🎉 

We now instructed our model on how to **request** that a tool be invoked.

Now, let's create some logic to actually run the tool!


<br>

## Invoking the tool 🏃

Now that the model can request that a tool be invoked, we need to write a function that can actually invoke 
the tool.

The function will select the appropriate tool by name, and pass to it the arguments chosen by the model.

In [9]:
from typing import Any, Dict, Optional, TypedDict

from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""

    name: str
    arguments: Dict[str, Any]


def invoke_tool(
    tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None
):
    """A function that we can use the perform a tool invocation.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool
    """
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

<br>

Let's test this out 🧪!

In [10]:
invoke_tool({"name": "multiply", "arguments": {"x": 3, "y": 5}})

15.0

<br>

## Let's put it together

Let's put it together into a chain that creates a calculator with add and multiplication capabilities.

In [11]:
chain = prompt | model | JsonOutputParser() | invoke_tool
chain.invoke({"input": "what's thirteen times 4.14137281"})

53.83784653

<br>

## Returning tool inputs

It can be helpful to return not only tool outputs but also tool inputs. We can easily do this with LCEL by `RunnablePassthrough.assign`-ing the tool output. This will take whatever the input is to the RunnablePassrthrough components (assumed to be a dictionary) and add a key to it while still passing through everything that's currently in the input:

In [12]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)
)
chain.invoke({"input": "what's thirteen times 4.14137281"})

{'name': 'multiply',
 'arguments': {'x': 13, 'y': 4.14137281},
 'output': 53.83784653}

<br>
<hr>
<br>
<br>

## 🚀 Your turn

Time to make this lab your own!

This how-to guide shows the "happy path" when the model correctly outputs all the required tool information. In reality, if you're using more complex tools, you may start encountering errors from the model, especially for models that have not been fine tuned for tool calling and for less capable models.

Replace the `multiply` and `add` tools with **3 new tools** of your choosing. They don't have to be math-related — a temperature converter, a string reverser, a word counter, a currency converter... pick whatever sounds fun to build.

<br>


Here's what to do:

1. **Design your 3 tools** — Write 3 new functions decorated with `@tool`, each with a clear docstring (this becomes the tool's description shown to the model) and typed arguments.
2. **Update the tools list** — Replace `tools = [multiply, add]` with your 3 new tools.
3. **Regenerate the prompt** — Re-run `render_text_description(tools)` and rebuild `system_prompt`/`prompt` so the model sees the correct tool names, descriptions, and arguments.
4. **Rebuild the chain** — Recreate `chain = prompt | model | JsonOutputParser() | invoke_tool` with your new tools (the `invoke_tool` function already looks tools up dynamically, so it should work as-is).
5. **Put it to the test** — Ask your chain several natural-language questions, one for each tool, and confirm it picks the right tool with the right arguments every time.
6. **Reflect** — In a markdown cell, briefly note any cases where the model picked the wrong tool or malformed the arguments, and why you think that happened.

<br>


💡 **Tip:**

- Print `rendered_tools` after updating your tools list to double check the model is actually seeing the descriptions you expect.

<br>


⭐️ **Bonus ideas:**

- Add a 4th tool that takes more than 2 arguments, or a mix of types (e.g. `str` and `int`), and see if the model still handles it correctly.
- Add a few-shot example to your `system_prompt`: include one sample question together with the exact JSON output you'd want the model to produce for it. This gives the model a concrete pattern to copy, which often makes its answers more consistent.
- Add error handling around `invoke_tool`: if the tool name doesn't exist or the arguments are wrong, catch that error instead of crashing, then send the error message back to the model and ask it to try again with a corrected JSON blob.
- Ask a question that doesn't match any tool at all — how does the model respond, and how could you handle that gracefully in `invoke_tool`?


## Solution: three text and temperature tools

The original calculator demo above is retained. Below I replace its tool registry with a temperature converter, a string reverser, and a word counter, then regenerate the descriptions and prompt. This run uses the installed `qwen3:4b`. Although that model also supports native tools, this notebook deliberately uses only text generation: the prompt requests a JSON tool call, LangChain parses it, and Python dispatches it. Ollama's JSON mode constrains syntax; it does not choose the tool or execute it.

In [13]:
import re
from typing import Literal

@tool
def convert_temperature(value: float, from_unit: Literal["C", "F"], to_unit: Literal["C", "F"]) -> float:
    """Convert a temperature between Celsius (C) and Fahrenheit (F). Use C or F for units."""
    if from_unit == to_unit:
        return float(value)
    if from_unit == "C":
        return value * 9 / 5 + 32
    return (value - 32) * 5 / 9

@tool
def reverse_text(text: str) -> str:
    """Reverse the characters of text exactly, preserving spaces and punctuation."""
    return text[::-1]

@tool
def count_words(text: str) -> int:
    """Count whitespace-separated words in text; punctuation stays attached to words."""
    return len(text.split())

tools = [convert_temperature, reverse_text, count_words]
rendered_tools = render_text_description(tools)
print(rendered_tools)
for selected_tool in tools:
    print(selected_tool.name, json.dumps(selected_tool.args, ensure_ascii=False))

convert_temperature(value: float, from_unit: Literal['C', 'F'], to_unit: Literal['C', 'F']) -> float - Convert a temperature between Celsius (C) and Fahrenheit (F). Use C or F for units.
reverse_text(text: str) -> str - Reverse the characters of text exactly, preserving spaces and punctuation.
count_words(text: str) -> int - Count whitespace-separated words in text; punctuation stays attached to words.
convert_temperature {"value": {"title": "Value", "type": "number"}, "from_unit": {"enum": ["C", "F"], "title": "From Unit", "type": "string"}, "to_unit": {"enum": ["C", "F"], "title": "To Unit", "type": "string"}}
reverse_text {"text": {"title": "Text", "type": "string"}}
count_words {"text": {"title": "Text", "type": "string"}}


In [14]:
system_prompt = """You select exactly one tool for a user request.
Available tools and their argument schemas:
""" + "\n".join(
    f"{t.name}: {t.description} Arguments: {json.dumps(t.args)}" for t in tools
) + """
Return only one JSON object with exactly two keys: "name" and "arguments".
Use an available tool name and the exact argument names from its schema.
Preserve quoted input text exactly. Temperature units must be C or F.
For requests unsupported by these tools, return {"name": "unsupported", "arguments": {}}.
Example user: Reverse the text "cat".
Example response: {"name": "reverse_text", "arguments": {"text": "cat"}}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", "{tool_instructions}"), ("user", "{input}")
]).partial(tool_instructions=system_prompt)

# Keep the lab's dynamic dispatcher, adding validation and graceful rejection.
def invoke_tool(tool_call_request: ToolCallRequest, config=None):
    if not isinstance(tool_call_request, dict):
        raise ValueError("Tool call must be an object")
    if set(tool_call_request) != {"name", "arguments"}:
        raise ValueError("Tool call requires exactly name and arguments")
    registry = {t.name: t for t in tools}
    name = tool_call_request["name"]
    if not isinstance(name, str) or not isinstance(tool_call_request["arguments"], dict):
        raise ValueError("Expected a string name and an arguments object")
    if name == "unsupported":
        return {"error": "No available tool can answer this request."}
    if name not in registry:
        raise ValueError(f"Unknown tool: {name}")
    selected = registry[name]
    if set(tool_call_request["arguments"]) != set(selected.args):
        raise ValueError(f"Expected arguments: {list(selected.args)}")
    return selected.invoke(tool_call_request["arguments"], config=config)

chain = prompt | model | JsonOutputParser() | invoke_tool
inspect_chain = (
    prompt | model | JsonOutputParser()
    | RunnablePassthrough.assign(output=invoke_tool)
)
print("Rebuilt prompt and chains for:", [t.name for t in tools])

Rebuilt prompt and chains for: ['convert_temperature', 'reverse_text', 'count_words']


In [15]:
import math

# Direct checks isolate tool correctness from model routing.
assert convert_temperature.invoke({"value": 0, "from_unit": "C", "to_unit": "F"}) == 32
assert convert_temperature.invoke({"value": 32, "from_unit": "F", "to_unit": "C"}) == 0
assert convert_temperature.invoke({"value": -40, "from_unit": "C", "to_unit": "F"}) == -40
assert reverse_text.invoke({"text": "Hello!"}) == "!olleH"
assert reverse_text.invoke({"text": ""}) == ""
assert count_words.invoke({"text": "  one\t two\nthree  "}) == 3
assert count_words.invoke({"text": ""}) == 0
print("7 direct tool checks passed.")

cases = [
    ("Convert 25 degrees Celsius to Fahrenheit.", "convert_temperature", {"value": 25, "from_unit": "C", "to_unit": "F"}, 77.0),
    ("Convert 212 degrees Fahrenheit to Celsius.", "convert_temperature", {"value": 212, "from_unit": "F", "to_unit": "C"}, 100.0),
    ('Reverse the text "Hello, lab!".', "reverse_text", {"text": "Hello, lab!"}, "!bal ,olleH"),
    ('Count the words in "Tools make language models useful".', "count_words", {"text": "Tools make language models useful"}, 5),
]
results = []
for question, expected_name, expected_arguments, expected_output in cases:
    result = inspect_chain.invoke({"input": question})
    print(question)
    print(json.dumps(result, ensure_ascii=False))
    assert result["name"] == expected_name, result
    assert result["arguments"] == expected_arguments, result
    if isinstance(expected_output, float):
        assert math.isclose(result["output"], expected_output, abs_tol=1e-9), result
    else:
        assert result["output"] == expected_output, result
    results.append(result)
print(f"All {len(results)} model-selected tool calls passed.")

7 direct tool checks passed.


Convert 25 degrees Celsius to Fahrenheit.
{"name": "convert_temperature", "arguments": {"value": 25, "from_unit": "C", "to_unit": "F"}, "output": 77.0}


Convert 212 degrees Fahrenheit to Celsius.
{"name": "convert_temperature", "arguments": {"value": 212, "from_unit": "F", "to_unit": "C"}, "output": 100.0}


Reverse the text "Hello, lab!".
{"name": "reverse_text", "arguments": {"text": "Hello, lab!"}, "output": "!bal ,olleH"}


Count the words in "Tools make language models useful".
{"name": "count_words", "arguments": {"text": "Tools make language models useful"}, "output": 5}
All 4 model-selected tool calls passed.


In [16]:
unsupported = inspect_chain.invoke({"input": "Who wrote Pride and Prejudice?"})
print(json.dumps(unsupported))
assert unsupported["name"] == "unsupported"
assert "error" in unsupported["output"]

# Deliberately malformed calls test the dispatch boundary, not model output.
from pydantic import ValidationError
invalid_calls = [
    {"name": "missing_tool", "arguments": {}},
    {"name": "reverse_text", "arguments": {}},
    {"name": "convert_temperature", "arguments": {"value": 10, "from_unit": "K", "to_unit": "C"}},
]
for invalid in invalid_calls:
    try:
        invoke_tool(invalid)
    except (ValueError, ValidationError) as error:
        print(type(error).__name__, str(error).splitlines()[0])
    else:
        raise AssertionError(f"Invalid call was accepted: {invalid}")
print("Unsupported request and invalid-call checks passed.")

{"name": "unsupported", "arguments": {}, "output": {"error": "No available tool can answer this request."}}
ValueError Unknown tool: missing_tool
ValueError Expected arguments: ['text']
ValidationError 1 validation error for convert_temperature
Unsupported request and invalid-call checks passed.


## Reflection

In this run, all four natural-language examples selected the correct tool and supplied the expected arguments; no malformed model calls were observed. The unsupported question was correctly rejected. All seven direct tool checks and all three injected invalid-call checks also passed. The examples check both directions of conversion, exact preservation of text, and word counting. Assertions compare the name, arguments, and result, so a plausible but incorrect answer cannot silently pass.

The explicit schemas, low temperature, JSON format constraint, and few-shot reversal example reduce ambiguity. JSON syntax alone does not guarantee a valid tool name, correct argument types, or the right interpretation of a request. The dispatcher therefore validates tool names and arguments, while the tool schema validates types and temperature units. The deliberately invalid calls are injected validation tests, not mistakes attributed to the model.

An unsupported request uses a sentinel and returns a clear error instead of guessing an unrelated tool. More ambiguous requests, multi-step tasks, or a smaller model could still produce incorrect routing. In a larger application I would add a bounded retry with validation feedback and collect ambiguous examples for evaluation. Reversing here operates on Python Unicode code points, and counting uses whitespace rather than linguistic tokenization.